In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

In [9]:
# 设置随机种子
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [10]:
set_random_seed(seed=42)

In [11]:
# 数据预处理和加载
train_transform = transforms.Compose([
    transforms.RandomRotation(10), # # 随机旋转图片，角度在-10到+10之间随机
    transforms.RandomHorizontalFlip(), # 随机水平翻转图片，概率为0.5（默认）
    transforms.RandomResizedCrop(148), # 随机裁剪并调整大小到148x148像素
    transforms.ToTensor(), # 转换为张量，并归一化到[0, 1]范围
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))# 使用均值和标准差对张量进行标准化
])

val_transform = transforms.Compose([
    transforms.Resize((148, 148)),  
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# 加载训练数据集和验证数据集，并应用定义的预处理变换
train_dataset = datasets.ImageFolder(root='./data/train', transform=train_transform)
val_dataset = datasets.ImageFolder(root='./data/validation', transform=val_transform)

# 创建训练数据加载器和验证数据加载器，shuffle=True数据随机打乱，shuffle=False保持数据顺序不变
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [12]:
# 模型训练函数
def train_model(model, criterion, optimizer, num_epochs, train_loader, val_loader, model_name):
    """
    训练给定的神经网络模型，并在每个epoch后验证模型性能。

    参数:
        model (torch.nn.Module): 要训练的模型。
        criterion: 损失函数。
        optimizer: 优化器。
        num_epochs (int): 训练轮数。
        train_loader: 训练数据加载器。
        val_loader: 验证数据加载器。
        model_name (str): 模型名称，用于保存权重和结果文件。
    """

    best_accuracy = 0.0  # 初始化最佳准确率为0
    best_model_path = f'./weights/{model_name}_best_model.pth'  # 定义保存最佳模型路径
    final_model_path = f'./weights/{model_name}_final_model.pth'  # 定义保存最终模型路径
    results_file = f'./training_results/{model_name}_training_results.txt'  # 定义训练结果记录文件路径

    # 创建结果文件
    os.makedirs('./training_results', exist_ok=True)
    os.makedirs('./weights', exist_ok=True)

    with open(results_file, 'w') as f:
        f.write('epoch,train_losses,train_accuracies,val_losses,val_accuracies\n')

    # 根据可用性选择计算设备（GPU或CPU）
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(num_epochs):
        model.train()  # 设置模型为训练模式
        running_loss = 0.0  # 初始化运行损失为0
        correct_train = 0  # 初始化正确预测数量为0
        total_train = 0  # 初始化总样本数量为0
        
        # 使用tqdm显示训练进度
        with tqdm(train_loader, unit="batch") as tepoch:
            for inputs, labels in tepoch:
                tepoch.set_description(f"Epoch {epoch + 1}/{num_epochs}")
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad() # 清除梯度
                
                outputs = model(inputs) # 前向传播
                loss = criterion(outputs, labels) # 计算损失
                loss.backward() # 反向传播
                optimizer.step() # 更新参数
                
                running_loss += loss.item() * inputs.size(0)  # 累加损失
                _, predicted = torch.max(outputs.data, 1)  # 获取预测值
                total_train += labels.size(0)  # 累加总样本数
                correct_train += (predicted == labels).sum().item()  # 累加正确预测数
                tepoch.set_postfix(loss=loss.item(), accuracy=100 * correct_train / total_train)  # 更新进度条信息
        
        # 计算平均训练损失和准确率
        train_loss = running_loss / len(train_loader.dataset)
        train_accuracy = 100 * correct_train / total_train
        
        # 验证模型
        model.eval() # 设置模型为评估模式
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            with tqdm(val_loader, unit="batch") as tepoch:
                for images, labels in tepoch:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                    running_val_loss += loss.item() * images.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    total_val += labels.size(0)
                    correct_val += (predicted == labels).sum().item()
                    tepoch.set_postfix(loss=loss.item(), accuracy=100 * correct_val / total_val)
        
        val_loss = running_val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct_val / total_val
        
        # 写入结果文件
        with open(results_file, 'a') as f:
            f.write(f'{epoch + 1},{train_loss:.4f},{0.01 * train_accuracy:.4f},{val_loss:.4f},{0.01 * val_accuracy:.4f}\n')
        
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {0.01 * train_accuracy:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {0.01 * val_accuracy:.4f}')
        
        # 保存最佳模型
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), best_model_path)
            print(f'Saved best model with accuracy: {val_accuracy:.2f}% at epoch {epoch + 1}')
    
    # 保存最终模型
    torch.save(model.state_dict(), final_model_path)
    print(f'Saved final model at epoch {num_epochs}')

In [13]:
# 由于LeNet5模型没有单独封装好的库可以直接调用，故自定义LeNet5模型
class LeNet5(nn.Module):
    def __init__(self, num_classes=2):
        super(LeNet5, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),  # 输入通道3，输出通道6，卷积核大小5
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),  # 池化大小2，步长2
            nn.Conv2d(6, 16, kernel_size=5),  # 输入通道6，输出通道16，卷积核大小5
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2)  # 池化大小2，步长2
        )
        # 手动计算特征图尺寸 (148-4)/2 -> (72-4)/2 = 34
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 34 * 34, 120),  # 16是通道数，34是特征图大小
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

In [14]:
# LeNet-5 模型训练
def train_lenet5():
    model = LeNet5(num_classes=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    train_model(model, criterion, optimizer, num_epochs=20, train_loader=train_loader, val_loader=val_loader, model_name='lenet5')

In [ ]:
# 开始训练LeNet-5 模型
train_lenet5()

In [7]:
# AlexNet 模型
def train_alexnet():
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, 2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    train_model(model, criterion, optimizer, num_epochs=20, train_loader=train_loader, val_loader=val_loader, model_name='alexnet')

In [ ]:
# 开始训练AlexNet 模型
train_alexnet()

In [ ]:
# ResNet18 模型
def train_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    train_model(model, criterion, optimizer, num_epochs=20, train_loader=train_loader, val_loader=val_loader, model_name='resnet18')

In [ ]:
# 开始训练ResNet18 模型
train_resnet18()

In [ ]:
# VGG模型
def train_vgg11():
    model = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
    print("Original classifier:", model.classifier)
    num_ftrs = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(num_ftrs, 2)  # 修改为 2 个输出类别
    print("Modified classifier:", model.classifier)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    train_model(model, criterion, optimizer, num_epochs=12, train_loader=train_loader, val_loader=val_loader, model_name='vgg11')

In [ ]:
# 开始训练VGG模型
train_vgg11()

In [ ]:
# densenet121模型
def train_densenet121():
    model = models.densenet121(pretrained=True)
    print("Original classifier:", model.classifier)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, 2)  # 修改为2个输出类别
    print("Modified classifier:", model.classifier)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    train_model(model, criterion, optimizer, num_epochs=12, train_loader=train_loader, val_loader=val_loader, model_name='densenet121')

In [ ]:
# 开始训练densenet121模型
train_densenet121()